# NB1 – Retail-Daten kennenlernen · Teil A: Datenexploration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muehlburger/bid-ws2026/blob/main/notebooks/BID_NB1_TeilA.ipynb) · Datensatz und Notebook: https://github.com/muehlburger/bid-ws2026
**Big Data (BID) · WS 2026/27 · Termin 1 · Aufgabe 3 · Paare · 25 Minuten**

**So arbeitet ihr**

1. **Zelle 1–3 einfach ausführen** (▶ links neben der Zelle) und lesen – sie laden den Datensatz und zeigen, wie er aufgebaut ist.
2. Danach **F1–F3** bearbeiten. In jeder Aufgabenzelle steht der Rechenweg fertig da; nur die mit `___` markierten Stellen ergänzt ihr (der Hinweis darüber sagt, womit).
3. Wer fertig ist: **Zusatzaufgabe** (Preisverteilung).

Regel im Raum: Wenn bei euch etwas nicht lädt, arbeitet beim Nachbarpaar mit – Setup-Probleme lösen wir nicht im Plenum.

**Der Datensatz** ist vollständig synthetisch (keine Kundendaten), aber im Schema eines typischen Warenwirtschafts-Exports: 12 Filialen (Filiale 1 = Onlineshop), 800 Artikelvarianten, 1.7.2024–30.6.2026, rund 400.000 Zeilen. **Eine Zeile = Tagesumsatz einer Variante in einer Filiale.**

In [ ]:
# Zelle 1 – Datensatz laden (einfach ausführen)
import os, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/muehlburger/bid-ws2026/main/data/clean/"  # öffentliches Repo mit dem synthetischen Datensatz (Basis-URL)
DATEIEN = ["variantenumsatz.parquet", "Filiale.csv"]

def hole(name):
    """Sucht die Datei zuerst neben dem Notebook, dann unter DATA_URL, sonst Upload-Dialog (nur in Colab)."""
    if os.path.exists(name):
        return name
    if not DATA_URL.startswith("https://<"):
        urllib.request.urlretrieve(DATA_URL + name, name)
        return name
    try:
        from google.colab import files  # gibt es nur in Colab
        print(f"Bitte {name} aus Moodle hochladen:")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"{name} nicht gefunden – Datei aus Moodle neben das Notebook legen.")

df = pd.read_parquet(hole("variantenumsatz.parquet"))
print("Zeilen und Spalten:", df.shape)
df.head(3)

In [ ]:
# Zelle 2 – Zeitraum und Umfang (einfach ausführen)
df["datum"] = pd.to_datetime(df["date_bestand_ums"])
print("Zeitraum:", df["datum"].min().date(), "bis", df["datum"].max().date())
print("Filialen:", df["filialnummer3"].nunique(), "| Varianten:", df["variantennummer8"].nunique(),
      "| Tage mit Umsatz:", df["datum"].nunique())
df.dtypes

**Eine Zeile erklärt** (die wichtigsten Spalten):

| Spalte | Bedeutung |
|---|---|
| `date_bestand_ums` | Buchungstag |
| `variantennummer8` | Artikelvariante (Artikel × Größe × Farbe) |
| `filialnummer3` | Filiale (1 = Onlineshop) |
| `verkaufspreis_ifw` | regulärer (Listen-)Verkaufspreis je Stück in € |
| `umsatz_menge_mk9` | verkaufte Stück – **negativ = Retoure** |
| `umsatz_wert_ifw` | Umsatz in € = Menge × regulärer Preis − Reduktionen (`redukt_*`); bezahlter Stückpreis = Umsatz ÷ Menge |
| `kz_aktion` | `A` = Aktions-/Sale-Verkauf, `R` = regulär |
| `kz_saison6` | Saison: `202531` = Frühjahr/Sommer 2025, `202532` = Herbst/Winter 2025 |

In [ ]:
# Zelle 3 – Stammdaten: Was steckt hinter einer Filialnummer? (einfach ausführen)
filialen = pd.read_csv(hole("Filiale.csv"), sep=";", decimal=",")
df = df.merge(filialen[["filialnummer3", "filial_bezeichnung", "hw_online"]], on="filialnummer3", how="left")
df.groupby(["filialnummer3", "filial_bezeichnung"])["umsatz_menge_mk9"].sum().sort_values(ascending=False)

## F1 – Skalenniveau und Diagrammwahl

Bestimme das **Skalenniveau** (nominal / ordinal / metrisch) von `filialnummer3`, `verkaufspreis_ifw`, `kz_aktion` und `date_bestand_ums`.
Welches **Diagramm** eignet sich für (a) Absatzmenge je Wochentag, (b) Verteilung des Verkaufspreises, (c) Anteil Aktions- vs. Normalverkäufe?

Führt die Zelle aus, schaut euch Typen und Beispielwerte an und schreibt die Antwort in die Textzelle darunter (Doppelklick zum Bearbeiten).

In [ ]:
# F1 – Datentypen und Beispielwerte ansehen (Achtung: der Datentyp ist nicht das Skalenniveau!)
spalten = ["filialnummer3", "verkaufspreis_ifw", "kz_aktion", "date_bestand_ums"]
print(df[spalten].dtypes)
df[spalten].sample(5, random_state=1)

**Eure Antwort F1**

- `filialnummer3`: …
- `verkaufspreis_ifw`: …
- `kz_aktion`: …
- `date_bestand_ums`: …
- Diagramme: (a) … · (b) … · (c) …

## F2 – Median vs. Mittelwert

Berechne **Median und Mittelwert der Tagesmenge je Variante und Filiale** (nur Verkäufe, Menge > 0).
Warum unterscheiden sie sich – und welche Kenngröße würdest du für „typischer Tagesabsatz“ berichten?

*Hinweis:* Die Tagesmenge je Variante und Filiale ist die Summe je `variantennummer8`, `filialnummer3` und `datum`. Für die beiden Kenngrößen gibt es die Methoden `.median()` und `.mean()`.

In [ ]:
# F2 – Tagesmenge je Variante × Filiale
verkaeufe = df[df["umsatz_menge_mk9"] > 0]                                   # nur Verkäufe (Retouren sind negativ)
tagesmenge = verkaeufe.groupby(["variantennummer8", "filialnummer3", "datum"])["umsatz_menge_mk9"].sum()

print("Median:     ", tagesmenge.___())          # ← ergänzen
print("Mittelwert: ", round(tagesmenge.___(), 2))   # ← ergänzen

tagesmenge.clip(upper=10).plot.hist(bins=range(1, 12), rwidth=0.9, title="Tagesmenge je Variante × Filiale (Stück, ≥10 zusammengefasst)")
plt.xlabel("Stück pro Tag"); plt.ylabel("Anzahl Verkaufstage"); plt.show()

**Eure Antwort F2:** Median = … · Mittelwert = … · Grund für den Unterschied: … · Für „typischer Tagesabsatz“ berichten wir: …

## F3 – Wochentagsprofil

Erstelle das **Wochentagsprofil der Gesamtmenge**. Welcher Tag ist der stärkste, welcher zeigt null – und was bedeutet das für eine Absatzprognose?

*Hinweis:* `dt.dayofweek` liefert 0 = Montag … 6 = Sonntag. Schaut euch zuerst das Ergebnis der Gruppierung an: **Welcher Wochentag fehlt – und warum?** Mit `.reindex(range(7), fill_value=0)` macht ihr alle sieben Tage sichtbar – überlegt dabei, woher ihr wisst, dass `0` die richtige Füllung ist.

In [ ]:
# F3 – Wochentagsprofil der Gesamtmenge
df["wochentag"] = df["datum"].dt.dayofweek                       # 0 = Montag … 6 = Sonntag
profil = df.groupby("wochentag")["umsatz_menge_mk9"].sum()
print(profil)                                                    # ← Welcher Tag fehlt hier? Warum?

profil = profil.___(range(7), fill_value=0)                      # ← ergänzen: alle sieben Wochentage sichtbar machen
profil.index = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
profil.plot.bar(title="Absatzmenge je Wochentag (Stück)", rot=0); plt.ylabel("Stück"); plt.show()

**Eure Antwort F3:** stärkster Tag = … · zweitstärkster = … · null am … weil … · Für die Prognose bedeutet das: …

## Zusatz (für schnelle Paare) – Verteilung des Verkaufspreises

Zeichne die Verteilung des regulären Verkaufspreises und benenne Preislagen. Welche Kenngröße beschreibt den „typischen Preis“ besser – Median oder Mittelwert?

In [ ]:
# Zusatz – Verteilung des Verkaufspreises (läuft ohne Ergänzung)
preise = df["verkaufspreis_ifw"]
print("Median:", preise.median(), "| Mittelwert:", round(preise.mean(), 2), "| Minimum:", preise.min(), "| Maximum:", preise.max())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
preise.plot.hist(bins=40, ax=ax[0], title="Histogramm Verkaufspreis (€)")
preise.plot.box(ax=ax[1], title="Boxplot Verkaufspreis (€)")
plt.show()
print("Die zehn häufigsten Preise:")
print(preise.round(2).value_counts().head(10))